# Incerteza e sensibilidade

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/A-incerteza.ipynb)

> **Objetivos.** Ao final deste apêndice você será capaz de:
> 1. representar dados incertos por distribuições de probabilidade e propagá-los por um modelo com o método de Monte Carlo;
> 2. estimar médias e probabilidades com o seu erro padrão e intervalos de confiança, e dimensionar o número de amostras;
> 3. usar a propagação linear (método delta) e reconhecer quando ela falha;
> 4. calcular sensibilidades por diferenças finitas e por diferenciação automática (PyTorch), e saber o custo de cada uma;
> 5. desconfiar de "o modelo nos valores médios" como resposta a uma pergunta de risco.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
import torch
from scipy.integrate import solve_bvp
from scipy.optimize import brentq
from cnlivro.incerteza import monte_carlo, probabilidade, propagacao_linear, gradiente_torch

## 1. Problema de engenharia: a pilha de bagaço vai pegar fogo?

Uma usina estoca bagaço de cana em pilhas ao ar livre durante a entressafra. A oxidação lenta do material libera calor, que só sai pela superfície. Se a pilha for grande demais, ou o dia quente demais, o calor gerado não consegue sair e a pilha entra em **combustão espontânea**. Já vimos o modelo no capítulo 14 (seção 7.1): numa pilha em forma de placa, de meia-espessura $L$, a elevação de temperatura adimensional $\theta$ segue o problema de Frank-Kamenetskii,

$$
\theta'' = -\delta\,e^{\theta}, \qquad \theta(-1) = \theta(1) = 0,
\qquad
\delta = \frac{E}{R T_0^2}\,\frac{Q\,\rho\,A\,L^2}{k}\,e^{-E/(R T_0)},
$$ (eq-fk)

com $T_0$ a temperatura ambiente, $k$ a condutividade do bagaço e $A$, $E$ e $Q$ os parâmetros da reação. Há solução estacionária (a pilha se estabiliza morna) só se $\delta < \delta_c \approx 0{,}878$. Acima disso, não há equilíbrio: a pilha esquenta até pegar fogo.

O problema de engenharia é que quase nada nessa fórmula é conhecido com precisão:

| Grandeza | Valor nominal | Incerteza (modelo adotado) |
| --- | --- | --- |
| meia-espessura da pilha $L$ | 2,5 m | uniforme entre 2,2 e 2,8 m (a pilha não é regular) |
| temperatura ambiente $T_0$ | 30 °C | normal, desvio-padrão 3 °C (variação entre dias de verão) |
| condutividade $k$ | 0,08 W/(m K) | normal, desvio de 15 % (umidade, compactação) |
| fator da taxa de reação $a = A/A_{\text{ref}}$ | 1 | lognormal: dentro de um fator 2 com 95 % de probabilidade |

Com os valores nominais, $\delta = 0{,}45$, bem abaixo do crítico. A pergunta é: **qual é a probabilidade de a pilha pegar fogo**, dadas essas incertezas?

In [ ]:
E_R, T_ref = 11_000.0, 303.15          # E/R (K) e temperatura de referência (K)

def delta_pilha(L, T0, k, a):
    # parâmetro de Frank-Kamenetskii, escrito em relação ao caso nominal (δ = 0,45)
    T = T0 + 273.15
    return 0.45 * (L / 2.5) ** 2 * (0.08 / k) * a * (T_ref / T) ** 2 * np.exp(E_R * (1 / T_ref - 1 / T))

# ponto crítico: δ(b) = 2b²/cosh²(b) tem máximo em tanh(b) = 1/b
b_c = brentq(lambda b: np.tanh(b) - 1 / b, 1, 1.5)
delta_c = 2 * b_c**2 / np.cosh(b_c) ** 2

def theta_centro(delta):
    # solução exata do ramo estável: θ(x) = 2 ln[cosh(b)/cosh(bx)], com δ = 2b²/cosh²(b), b < b_c.
    # Sem solução (δ > δ_c), devolve inf.
    delta = np.asarray(delta, dtype=float)
    lo, hi = np.zeros_like(delta), np.full_like(delta, b_c)
    for _ in range(60):                               # bisseção vetorizada (capítulo 3)
        m = (lo + hi) / 2
        abaixo = 2 * m**2 / np.cosh(m) ** 2 < delta
        lo, hi = np.where(abaixo, m, lo), np.where(abaixo, hi, m)
    return np.where(delta < delta_c, 2 * np.log(np.cosh((lo + hi) / 2)), np.inf)

d_nom = delta_pilha(2.5, 30.0, 0.08, 1.0)
print(f"δ nominal = {d_nom:.3f};  δ crítico = {delta_c:.4f};  θ(0) nominal = {theta_centro(d_nom):.4f}")
print(f"elevação de temperatura no centro: ΔT = θ R T0²/E = {theta_centro(d_nom) * T_ref**2 / E_R:.2f} K")

No caso nominal, o centro da pilha fica só 2,4 K acima do ambiente. A resposta do modelo nos valores nominais é tranquilizadora. Este apêndice mostra por que ela não responde à pergunta.

## 2. A ideia em uma imagem

O **método de Monte Carlo** responde à pergunta da forma mais direta possível: sorteia muitos "mundos possíveis" (cada um com um valor de $L$, $T_0$, $k$ e $a$, tirado das suas distribuições), roda o modelo em cada um e conta em quantos a pilha pega fogo. A fração é a estimativa da probabilidade, e a sua incerteza cai com o número de sorteios. Arraste o slider para aumentar o número de amostras.

In [ ]:
rng = np.random.default_rng(2026)
N_max = 100_000

def sorteia(N, rng, L=(2.2, 2.8)):
    return dict(L=rng.uniform(*L, N), T0=rng.normal(30, 3, N), k=rng.normal(0.08, 0.012, N),
                a=np.exp(rng.normal(0, np.log(2) / 1.96, N)))

entradas = sorteia(N_max, rng)
deltas = delta_pilha(**entradas)
bordas = np.linspace(0, 2, 61)

def quadro_mc(N):
    d = deltas[:N]
    p, lo, hi = probabilidade(d > delta_c)
    fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.12,
                        subplot_titles=["δ sorteado (histograma)", "estimativa de P(fogo) com IC de 95 %"])
    cont, _ = np.histogram(d, bins=bordas)
    meio = (bordas[1:] + bordas[:-1]) / 2
    cores_b = [CORES["erro"] if m > delta_c else CORES["aproximacao"] for m in meio]
    fig.add_trace(go.Bar(x=meio, y=cont / N / (bordas[1] - bordas[0]), marker_color=cores_b, showlegend=False), 1, 1)
    fig.add_vline(x=delta_c, line=dict(color=CORES["exata"], dash="dash"), row=1, col=1)
    fig.add_annotation(x=delta_c, y=2.4, text="δ<sub>c</sub>: sem equilíbrio", showarrow=False, xanchor="left",
                       xshift=5, row=1, col=1)
    ns = np.array([n for n in [10, 30, 100, 300, 1000, 3000, 10_000, 30_000, 100_000] if n <= N])
    est = np.array([probabilidade(deltas[:n] > delta_c) for n in ns])
    fig.add_trace(go.Scatter(x=ns, y=est[:, 0] * 100, mode="lines+markers", showlegend=False,
                             line=dict(color=CORES["aproximacao"]),
                             error_y=dict(type="data", symmetric=False, array=(est[:, 2] - est[:, 0]) * 100,
                                          arrayminus=(est[:, 0] - est[:, 1]) * 100)), 1, 2)
    fig.update_xaxes(title="δ", range=[0, 2], row=1, col=1)
    fig.update_yaxes(title="densidade", range=[0, 2.8], row=1, col=1)
    fig.update_xaxes(title="número de amostras N", type="log", range=[0.8, 5.2], row=1, col=2)
    fig.update_yaxes(title="P(fogo) (%)", range=[0, 30], row=1, col=2)
    fig.update_layout(width=980, height=430, margin=dict(t=110), bargap=0.05,
                      title=f"N = {N}: P(fogo) = {numero(100 * p, 1)} %, IC de 95 % [{numero(100 * lo, 1)}; {numero(100 * hi, 1)}] %")
    return fig

quadros = [(str(n), quadro_mc(n)) for n in [100, 1000, 10_000, 100_000]]
mostrar(com_controle(quadros, rotulo="N = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** o histograma de $\delta$ é assimétrico, com uma cauda longa para a direita: o produto de vários fatores incertos, um deles exponencial em $T_0$, alarga a distribuição para cima. A parte vermelha, à direita de $\delta_c$, são os mundos em que a pilha pega fogo: cerca de **um em cada dez**. Com 100 amostras, a estimativa ainda é grosseira; com 100 mil, a probabilidade está determinada com uma casa decimal. Nenhum cálculo "no valor nominal" vê essa cauda.

## 3. Formulação matemática

### 3.1 Entradas incertas

Cada dado incerto vira uma **variável aleatória** com uma distribuição escolhida a partir do que se sabe:

- **normal** $\mathcal N(\mu, \sigma^2)$: erros de medida, variações simétricas em torno de um valor típico;
- **lognormal** ($\ln X$ normal): grandezas positivas conhecidas "a menos de um fator", como taxas de reação, permeabilidades, coeficientes de dispersão; "fator 2 com 95 %" significa $\sigma_{\ln} = \ln 2/1{,}96$;
- **uniforme** $\mathcal U(a, b)$: só se conhecem os limites.

A escolha da distribuição é uma hipótese de modelagem, como a escolha da equação, e afeta sobretudo as **caudas** (seção 7.3).

### 3.2 Monte Carlo

Com $N$ amostras independentes $\mathbf x^{(1)}, \dots, \mathbf x^{(N)}$ das entradas, as saídas $y^{(j)} = f(\mathbf x^{(j)})$ são uma amostra da distribuição de $Y = f(\mathbf X)$. A média e a probabilidade de um evento são estimadas por

$$
\bar y = \frac1N\sum_j y^{(j)}, \quad \text{erro padrão } \frac{s}{\sqrt N};
\qquad
\hat p = \frac{\#\{\text{eventos}\}}{N}, \quad \text{erro padrão } \sqrt{\frac{\hat p(1 - \hat p)}{N}}.
$$

O erro cai como $N^{-1/2}$, **qualquer que seja o número de entradas**, como na integração por Monte Carlo do capítulo 11 (e é a mesma coisa: $\mathbb E[f(\mathbf X)]$ é uma integral). Para ganhar uma casa decimal, 100 vezes mais amostras. Para uma probabilidade pequena, o que importa é o erro **relativo**, $\sqrt{(1 - p)/(Np)} \approx 1/\sqrt{\text{número de eventos}}$: para estimar $p$ com 10 % de erro, são precisos uns 100 eventos, ou seja, $N \approx 100/p$.

Para um intervalo de confiança de $p$, o intervalo de **Wilson** (função `probabilidade` do pacote) se comporta bem mesmo com poucos eventos, e não colapsa em $[0, 0]$ quando nenhum evento aparece: com $N$ amostras e zero eventos, o limite superior de 95 % é cerca de $3/N$ (a "regra do 3").

Amostragens mais espertas (hipercubo latino, sequências de quase Monte Carlo como as de Sobol, em `scipy.stats.qmc`) cobrem o espaço das entradas de modo mais uniforme e reduzem o erro para funções suaves. Para eventos raros, a **amostragem por importância** sorteia mais na região do evento e corrige os pesos.

### 3.3 Propagação linear (método delta)

Se as incertezas são pequenas, linearizamos $f$ em torno do valor nominal $\boldsymbol\mu$:

$$
Y \approx f(\boldsymbol\mu) + \sum_i \frac{\partial f}{\partial x_i}(\boldsymbol\mu)\,(X_i - \mu_i)
\quad\Longrightarrow\quad
\sigma_Y^2 \approx \sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2 \quad (\text{entradas independentes}).
$$

É barato ($n + 1$ ou $2n$ avaliações de $f$ com diferenças finitas) e diz quanto cada entrada contribui para a variância. Mas supõe que $f$ é aproximadamente linear **na faixa de variação das entradas** e devolve só média e desvio, como se a saída fosse normal. Duas consequências: $\mathbb E[f(\mathbf X)] \ne f(\mathbb E[\mathbf X])$ quando $f$ é curva (desigualdade de Jensen), e a linearização não enxerga limiares, saturações ou a falta de solução.

### 3.4 Sensibilidades: diferenças finitas e diferenciação automática

As derivadas $\partial f/\partial x_i$ medem a **sensibilidade local**. Para comparar entradas de unidades diferentes, usa-se a **elasticidade** $S_i = \frac{x_i}{f}\frac{\partial f}{\partial x_i}$: a variação percentual da saída por 1 % de variação da entrada. Três maneiras de calcular as derivadas:

| Método | Custo para $n$ entradas | Precisão | Cuidado |
| --- | --- | --- | --- |
| diferenças finitas (capítulo 10) | $n + 1$ ou $2n$ execuções do modelo | limitada pela escolha de $h$ | ruído do modelo (tolerâncias de solvers) é amplificado por $1/h$ |
| passo complexo (capítulo 10) | $n$ execuções em aritmética complexa | precisão de máquina | o código tem de aceitar números complexos |
| diferenciação automática, modo reverso | **uma** execução + um passo para trás | precisão de máquina | o código tem de ser escrito numa biblioteca que a suporte (PyTorch, JAX) |

A **diferenciação automática** aplica a regra da cadeia às operações elementares que o programa executa, registradas durante a execução. No **modo reverso** (o *backpropagation* das redes neurais do capítulo 9), as derivadas da saída em relação a **todas** as entradas saem de uma única passagem de trás para a frente, com custo de poucas avaliações de $f$, independentemente de $n$. Ela deriva o **programa**, inclusive laços, iterações de Newton e passos de tempo, e devolve a derivada exata do algoritmo discretizado.

Medidas **globais** de sensibilidade, como os índices de Sobol (a fração da variância de $Y$ explicada por cada entrada, considerando toda a faixa de variação), não dependem da linearização, mas custam muitas execuções de Monte Carlo.

## 4. À mão

**Propagação linear em $\ln\delta$.** Como $\delta$ é um produto de potências, $\ln\delta$ é uma soma, e a propagação linear em $\ln\delta$ fica simples. As derivadas de $\ln\delta$ no ponto nominal:

$$
\frac{\partial\ln\delta}{\partial L} = \frac{2}{L} = 0{,}8\ \text{m}^{-1}, \quad
\frac{\partial\ln\delta}{\partial T_0} = \frac{E/R}{T_0^2} - \frac{2}{T_0} = 0{,}1197 - 0{,}0066 = 0{,}113\ \text{K}^{-1}, \quad
\frac{\partial\ln\delta}{\partial k} = -\frac1k = -12{,}5, \quad
\frac{\partial\ln\delta}{\partial \ln a} = 1.
$$

Os desvios: $L$ uniforme em $[2{,}2; 2{,}8]$ tem $\sigma = 0{,}6/\sqrt{12} = 0{,}173$ m; $\sigma_{T_0} = 3$ K; $\sigma_k = 0{,}012$; $\sigma_{\ln a} = \ln 2/1{,}96 = 0{,}354$. As contribuições para a variância de $\ln\delta$:

$$
(0{,}8 \cdot 0{,}173)^2 + (0{,}113 \cdot 3)^2 + (12{,}5 \cdot 0{,}012)^2 + 0{,}354^2 = 0{,}019 + 0{,}115 + 0{,}023 + 0{,}125 = 0{,}282,
$$

logo $\sigma_{\ln\delta} \approx 0{,}53$. Se $\ln\delta$ fosse normal com média $\ln 0{,}45$, $P(\delta > \delta_c) = P\big(Z > \ln(0{,}878/0{,}45)/0{,}53\big) = P(Z > 1{,}26) \approx 10\ \%$. A temperatura ambiente e a taxa de reação respondem por mais de 80 % da variância.

**Quantas amostras?** Para estimar $p \approx 0{,}1$ com erro padrão de 0,5 ponto percentual: $\sqrt{0{,}1 \cdot 0{,}9/N} = 0{,}005 \Rightarrow N = 3\,600$. Para $p \approx 0{,}001$ com o mesmo erro **relativo** (5 %), $N \approx 400\,000$.

In [ ]:
x_nom = np.array([2.5, 30.0, 0.08, 1.0])
sig_nom = np.array([0.6 / np.sqrt(12), 3.0, 0.012, np.log(2) / 1.96])     # para a, o desvio de ln a
ln_delta = lambda L, T0, k, a: torch.log(0.45 * (L / 2.5) ** 2 * (0.08 / k) * a * (T_ref / (T0 + 273.15)) ** 2
                                        * torch.exp(E_R * (1 / T_ref - 1 / (T0 + 273.15))))
_, g = gradiente_torch(ln_delta, x_nom)          # derivadas exatas por diferenciação automática
contrib = (g * sig_nom) ** 2                      # (para a: ∂lnδ/∂a · σ_ln a, pois a = 1 no ponto nominal)
print("∂ ln δ/∂x (autograd):", np.round(g, 4))
print("contribuições para var(ln δ):", np.round(contrib, 3), " soma:", round(contrib.sum(), 3))

## 5. Explore

### 5.1 Quem manda no risco?

O gráfico de "tornado" mostra a contribuição de cada entrada para a variância de $\ln\delta$, e quanto a probabilidade de fogo cai se a incerteza daquela entrada for **eliminada** (fixando-a no valor nominal, por Monte Carlo):

In [ ]:
nomes = {"L": "meia-espessura L", "T0": "temperatura ambiente T₀", "k": "condutividade k", "a": "taxa de reação a"}
p_base = probabilidade(deltas > delta_c)[0]
p_sem = {}
for chave in nomes:
    ent = dict(entradas)
    ent[chave] = np.full(N_max, {"L": 2.5, "T0": 30.0, "k": 0.08, "a": 1.0}[chave])
    p_sem[chave] = probabilidade(delta_pilha(**ent) > delta_c)[0]
ordem = np.argsort(contrib)
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=["fração da variância de ln δ", "P(fogo) sem a incerteza desta entrada"])
rot = [list(nomes.values())[i] for i in ordem]
fig.add_trace(go.Bar(y=rot, x=(contrib / contrib.sum())[ordem] * 100, orientation="h", marker_color=CORES["aproximacao"],
                     text=[f"{numero(v, 0)} %" for v in (contrib / contrib.sum())[ordem] * 100], textposition="outside"), 1, 1)
fig.add_trace(go.Bar(y=rot, x=[p_sem[list(nomes)[i]] * 100 for i in ordem], orientation="h", marker_color=CORES["secundaria"],
                     text=[f"{numero(p_sem[list(nomes)[i]] * 100, 1)} %" for i in ordem], textposition="outside"), 1, 2)
fig.add_vline(x=p_base * 100, line=dict(color=CORES["erro"], dash="dash"), row=1, col=2)
fig.add_annotation(x=p_base * 100, y=-0.75, text=f"com todas as incertezas: {numero(p_base * 100, 1)} %", showarrow=False,
                   xanchor="right", xshift=-4, font=dict(color=CORES["erro"]), row=1, col=2)
fig.update_xaxes(title="%", range=[0, 60], row=1, col=1)
fig.update_xaxes(title="%", range=[0, 14], row=1, col=2)
fig.update_layout(width=980, height=420, showlegend=False, margin=dict(l=190, t=110),
                  title="A temperatura ambiente e a taxa de reação dominam o risco")
mostrar(fig)

**O que notar:** medir melhor a condutividade ou a geometria da pilha quase não muda o risco. Conhecer melhor a cinética do bagaço (por ensaios de laboratório) ou limitar a operação a dias menos quentes é o que reduz a incerteza. Essa é a utilidade prática da análise de sensibilidade: dizer **onde vale a pena gastar** em medições.

### 5.2 Diferenciação automática através de um solver

A diferenciação automática deriva qualquer programa escrito com operações do PyTorch. Aqui, ela deriva um solver completo: diferenças finitas para o problema {eq}`eq-fk` com 50 subintervalos e 25 iterações de Newton (capítulo 14), e devolve $d\theta(0)/d\delta$, que comparamos com a derivada da solução exata. Mude $\delta$ e veja o que acontece perto do ponto crítico.

In [ ]:
def theta0_torch(delta, n=50, iteracoes=25):
    # diferenças finitas + Newton, escritos em PyTorch: o autograd segue todas as iterações
    h = 2.0 / n
    m = n - 1
    D2 = (torch.diag(torch.full((m,), -2.0, dtype=torch.float64)) + torch.diag(torch.ones(m - 1, dtype=torch.float64), 1)
          + torch.diag(torch.ones(m - 1, dtype=torch.float64), -1)) / h**2
    th = torch.zeros(m, dtype=torch.float64)
    for _ in range(iteracoes):
        R = D2 @ th + delta * torch.exp(th)
        J = D2 + torch.diag(delta * torch.exp(th))
        th = th - torch.linalg.solve(J, R)
    return th[m // 2]

def deriv_exata(delta):
    lo, hi = 0.0, b_c
    for _ in range(80):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if 2 * mid**2 / np.cosh(mid) ** 2 < delta else (lo, mid)
    b = (lo + hi) / 2
    dg = 4 * b * (1 - b * np.tanh(b)) / np.cosh(b) ** 2       # dδ/db
    return 2 * np.tanh(b) / dg

d_grade = np.linspace(0.02, 0.875, 80)
ex_grade = np.array([deriv_exata(d) for d in d_grade])

def quadro_ad(d0):
    dt = torch.tensor(d0, dtype=torch.float64, requires_grad=True)
    th0 = theta0_torch(dt)
    (g_ad,) = torch.autograd.grad(th0, dt)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d_grade, y=ex_grade, mode="lines", name="dθ(0)/dδ exata", line=dict(color=CORES["exata"], dash="dash")))
    fig.add_trace(go.Scatter(x=[d0], y=[float(g_ad)], mode="markers", name="autograd através do solver",
                             marker=dict(color=CORES["aproximacao"], size=13)))
    fig.add_vline(x=delta_c, line=dict(color=CORES["erro"], dash="dot"))
    fig.update_xaxes(title="δ", range=[0, 0.92])
    fig.update_yaxes(title="dθ(0)/dδ", type="log", range=[-0.4, 1.6], tickvals=[0.5, 1, 2, 5, 10, 20],
                     ticktext=["0,5", "1", "2", "5", "10", "20"])
    fig.update_layout(width=780, height=420, legend=dict(x=0.02, y=0.98),
                      title=f"δ = {numero(d0, 2)}: autograd {numero(float(g_ad), 3)}, exata {numero(deriv_exata(d0), 3)} "
                            f"(θ(0) = {numero(float(th0.detach()), 3)})")
    return fig

quadros = [(numero(d, 2), quadro_ad(d)) for d in [0.2, 0.45, 0.7, 0.8, 0.85, 0.87]]
mostrar(com_controle(quadros, rotulo="δ = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** o autograd devolve a derivada do **programa** (as diferenças finitas com 50 subintervalos), que difere da derivada exata só pelo erro de discretização da malha, de menos de 1 %. E a sensibilidade **explode** perto de $\delta_c$: ali a solução estável encontra a instável e deixa de existir (uma dobra, como no capítulo 14). Nenhuma linearização em torno do ponto nominal ($\delta = 0{,}45$, onde a derivada é moderada) consegue prever esse comportamento.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"A temperatura de uma pilha de bagaço segue o modelo de Frank-Kamenetskii (equação e parâmetros anexos). Os parâmetros L, T0, k e o fator da taxa de reação são incertos (distribuições anexas). Calcule a elevação de temperatura no centro e a sua incerteza, e diga se a pilha é segura."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
import numpy as np
from scipy.integrate import solve_bvp

def theta_max(L, T0, k, a):
    d = delta_pilha(L, T0, k, a)
    x = np.linspace(-1, 1, 41)
    sol = solve_bvp(lambda x, y: np.vstack([y[1], -d * np.exp(y[0])]),
                    lambda ya, yb: np.array([ya[0], yb[0]]), x, np.zeros((2, x.size)), tol=1e-8)
    return sol.sol(0.0)[0]

nominal = np.array([2.5, 30.0, 0.08, 1.0])
desvios = np.array([0.6 / np.sqrt(12), 3.0, 0.012, np.log(2) / 1.96])

# propagação de incerteza de primeira ordem, com derivadas por diferenças centradas
th0 = theta_max(*nominal)
grad = np.zeros(4)
for i in range(4):
    dx = 1e-4 * nominal[i]
    xp, xm = nominal.copy(), nominal.copy()
    xp[i] += dx
    xm[i] -= dx
    grad[i] = (theta_max(*xp) - theta_max(*xm)) / (2 * dx)
sigma = np.sqrt(np.sum((grad * desvios) ** 2))

escala = T_ref**2 / E_R        # K por unidade de θ
print(f"θ(0) = {th0:.3f} ± {sigma:.3f}  ->  ΔT = {th0 * escala:.1f} ± {sigma * escala:.1f} K")
print(f"limite superior de 95 % (1,645σ): ΔT = {(th0 + 1.645 * sigma) * escala:.1f} K")
theta_c = 2 * np.log(np.cosh(1.1997))
print(f"θ crítico (Frank-Kamenetskii) = {theta_c:.3f}: margem de {(theta_c - th0) / sigma:.1f} desvios-padrão.")
print("Conclusão: a pilha é segura; mesmo no pior caso de 95 %, a elevação fica bem abaixo do valor crítico.")

O código é cuidadoso: resolve o problema de contorno com tolerância apertada, calcula as derivadas por diferenças centradas e propaga a incerteza pela fórmula da seção 3.3. A margem de 4,5 desvios-padrão parece confortável.

### 6.2 Verificação

**Monte Carlo com o mesmo modelo.** A solução exata `theta_centro` coincide com a do `solve_bvp` do agente no ponto nominal, e é vetorizada, o que permite usar as 100 mil amostras da seção 2:

In [ ]:
print(f"θ(0) nominal: solve_bvp do agente {th0:.6f}, solução exata {theta_centro(d_nom):.6f}")
th_mc = theta_centro(deltas)
fogo = ~np.isfinite(th_mc)
p, lo, hi = probabilidade(fogo)
print(f"amostras sem solução estacionária (fogo): {100 * p:.2f} %  (IC de 95 %: {100 * lo:.2f} a {100 * hi:.2f} %)")
estaveis = th_mc[~fogo]
print(f"entre as estáveis: θ(0) médio {estaveis.mean():.3f}, percentil 95 {np.quantile(estaveis, 0.95):.3f}, "
      f"máximo {estaveis.max():.3f}")

Em **10,5 %** dos cenários, a pilha não tem equilíbrio e pega fogo. A previsão do agente, "segura com margem de 4,5 desvios-padrão", errou por um fator de dezenas de milhares: numa normal, 4,5 desvios correspondem a uma chance de 3 em um milhão.

**O que o código do agente faz num desses cenários?**

In [ ]:
import warnings
for d_teste in [0.85, 0.95, 1.2]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        x = np.linspace(-1, 1, 41)
        sol = solve_bvp(lambda x, y: np.vstack([y[1], -d_teste * np.exp(y[0])]), lambda ya, yb: np.array([ya[0], yb[0]]),
                        x, np.zeros((2, 41)), tol=1e-8)
    print(f"δ = {d_teste}: solve_bvp devolve θ(0) = {sol.sol(0.0)[0]:.4g}, status {sol.status} ({sol.message})")

Acima de $\delta_c$, o `solve_bvp` não converge e diz isso no `status`, mas devolve um número mesmo assim. A função `theta_max` do agente descarta o status: se ela fosse usada num Monte Carlo, os cenários de fogo apareceriam como valores pequenos ou grandes sem nenhum aviso (seção 7.2).

### 6.3 Por quê

A propagação linear responde a uma pergunta diferente da que foi feita. Ela descreve as pequenas variações de $\theta(0)$ em torno do ponto nominal, onde o modelo é suave e a derivada é moderada. Mas o evento perigoso não é "$\theta(0)$ um pouco maior": é a **ausência de solução**, que acontece quando a combinação das entradas leva $\delta$ além de 0,878. Esse limiar está a 1,3 desvio-padrão de $\ln\delta$ do ponto nominal (seção 4), mas é invisível para a derivada em $\delta = 0{,}45$, que não sabe que a curva vai dobrar. O "$\theta_c$" usado pelo agente para a margem também não ajuda: o que limita o risco é o parâmetro $\delta$, e não o valor de $\theta$, porque perto da dobra $\theta(0)$ fica limitado (cerca de 1,19) enquanto a sua derivada vai a infinito (seção 5.2).

### 6.4 A correção

Monte Carlo, com o evento de interesse definido explicitamente (existência de solução), o intervalo de confiança da probabilidade e a análise de sensibilidade sobre o risco, não sobre a média. E uma proposta de engenharia: reduzir a altura da pilha.

In [ ]:
print("meia-espessura L (m)   P(fogo)   IC de 95 %")
for faixa in [(2.2, 2.8), (1.8, 2.2), (1.6, 2.0), (1.5, 1.8)]:
    d = delta_pilha(**sorteia(N_max, np.random.default_rng(7), L=faixa))
    p, lo, hi = probabilidade(d > delta_c)
    print(f"   {faixa[0]:.1f} a {faixa[1]:.1f}          {100 * p:5.2f} %   [{100 * lo:.2f}; {100 * hi:.2f}] %")

Pilhas com 3,6 a 4,4 m de espessura (meia-espessura de 1,8 a 2,2 m) reduzem o risco a menos de 2 %; para ficar perto de 0,2 %, menos de 3,6 m. A decisão entre essas opções (custo da área de estocagem contra risco de incêndio) é da engenharia, mas agora ela pode ser tomada com números.

> **Lição.** "O modelo nos valores médios, mais ou menos um desvio" não responde a uma pergunta de **risco**. Quando o modelo tem limiares, saturações, bifurcações ou falta de solução, peça ao agente **Monte Carlo** com o evento definido explicitamente, a probabilidade com **intervalo de confiança**, e a checagem do `status` de cada solução. Use a propagação linear como estimativa rápida da dispersão e das contribuições de cada entrada, e nunca como estimativa de uma cauda.

## 7. Quando falha

### 7.1 Eventos raros com poucas amostras

Com pilhas de 3,0 a 3,6 m (meia-espessura de 1,5 a 1,8 m), a probabilidade de fogo é de cerca de 0,24 %. Dez estudos de Monte Carlo independentes, cada um com 1 000 amostras:

In [ ]:
for semente in range(10):
    d = delta_pilha(**sorteia(1000, np.random.default_rng(100 + semente), L=(1.5, 1.8)))
    p, lo, hi = probabilidade(d > delta_c)
    print(f"estudo {semente + 1:2d}: {round(p * 1000)} eventos -> P = {100 * p:.1f} %, IC de 95 % [{100 * lo:.2f}; {100 * hi:.2f}] %")

As estimativas pontuais variam por um fator de várias vezes entre estudos idênticos. Com $p = 0{,}24$ % e $N = 1\,000$, cerca de 9 % dos estudos não veem **nenhum** evento ($e^{-2{,}4}$) e relatariam "risco zero"; o intervalo de Wilson, nesse caso, vai até 0,38 %, mostrando que zero eventos não prova risco zero. Para probabilidades de $10^{-3}$ ou menores, são precisas centenas de milhares de amostras, ou técnicas de amostragem por importância. Desconfie de um relatório que diz "nenhuma falha em 1 000 simulações" sem o intervalo.

### 7.2 Ignorar o status do solver

O código do agente, usado dentro de um laço de Monte Carlo, devolveria um número para cada cenário, inclusive para os sem solução (seção 6.2). Contando como "fogo" só os valores de $\theta(0)$ acima de, digamos, 1,19:

In [ ]:
amostra = {k_: v[:300] for k_, v in entradas.items()}
th_agente = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for i in range(300):
        th_agente.append(theta_max(*(amostra[k_][i] for k_ in ["L", "T0", "k", "a"])))
th_agente = np.array(th_agente)
fogo_verdade = delta_pilha(**amostra) > delta_c
print(f"cenários sem solução entre as 300 primeiras amostras: {fogo_verdade.sum()}")
print(f"o código do agente os classifica como: θ(0) entre {th_agente[fogo_verdade].min():.3g} e {th_agente[fogo_verdade].max():.3g}")
print(f"'fogo' detectado pelo critério θ(0) > 1,19: {np.sum(th_agente > 1.19)} cenários")

Menos da metade dos cenários de fogo é detectada; os outros aparecem com $\theta(0)$ pequeno, ou até negativo, como uma pilha morna (ou mais fria que o ambiente!), e o risco seria subestimado. Todo solver iterativo (raízes, sistemas não lineares, problemas de contorno, integradores de EDO) tem uma forma de dizer que falhou: `status`, `success`, `converged`, um aviso. Num Monte Carlo, que roda o solver milhares de vezes em cenários que ninguém olha um por um, **conferir e contar as falhas** é obrigatório.

### 7.3 A cauda depende da distribuição escolhida

O fator da taxa de reação foi modelado como lognormal (fator 2 com 95 %). Se ele for modelado como uniforme entre 0,5 e 1,5 (mesma mediana, cauda curta), ou se a temperatura for uniforme com o mesmo desvio-padrão:

In [ ]:
r_ = np.random.default_rng(11)
casos = {
    "como no texto (T0 normal, a lognormal)": sorteia(N_max, r_),
    "T0 uniforme, mesmo desvio": dict(sorteia(N_max, r_), T0=r_.uniform(30 - 3 * np.sqrt(3), 30 + 3 * np.sqrt(3), N_max)),
    "a uniforme entre 0,5 e 1,5": dict(sorteia(N_max, r_), a=r_.uniform(0.5, 1.5, N_max)),
}
for nome, ent in casos.items():
    d = delta_pilha(**ent)
    print(f"{nome:42s}: P(fogo) = {100 * probabilidade(d > delta_c)[0]:.1f} %,  δ mediano {np.median(d):.3f}")

Mudar a forma da distribuição, mantendo o valor central, muda a probabilidade: pouco no caso da temperatura (com o mesmo desvio), e quase um terço no caso da taxa de reação, cuja cauda superior encolhe. O risco mora na cauda, e a cauda é a parte da distribuição que os dados costumam conhecer pior. Um estudo de risco sério testa a sensibilidade da resposta às **hipóteses de distribuição**, e não só aos parâmetros.

## Resumo

| Técnica | O que dá | Custo | Quando usar | Cuidado |
| --- | --- | --- | --- | --- |
| modelo no valor nominal | um número | 1 execução | nunca como resposta de risco | ignora a variabilidade |
| propagação linear (método delta) | média e desvio aproximados, contribuições | $n + 1$ a $2n$ execuções | incertezas pequenas, modelo suave | não vê caudas, limiares, falta de solução |
| Monte Carlo | a distribuição da saída, probabilidades | $N$ execuções, erro $\propto N^{-1/2}$ | caso geral | eventos raros pedem $N \approx 100/p$; conferir falhas |
| hipercubo latino / quase Monte Carlo | o mesmo, com menos amostras | $N$ execuções | modelos suaves e caros | menos ganho em eventos raros |
| diferenças finitas | sensibilidades locais | $n + 1$ a $2n$ execuções | modelos de caixa-preta | escolha de $h$, ruído do solver |
| diferenciação automática | sensibilidades locais exatas do programa | ~1 execução, qualquer $n$ | modelos escritos em PyTorch/JAX | perto de bifurcações, a derivada explode |
| índices de Sobol | sensibilidade global | muitas execuções | modelos não lineares | custo |

## Exercícios

**Conceituais (sem computador)**

1. Mostre que, se $Y = X^2$ com $X \sim \mathcal N(\mu, \sigma^2)$, então $\mathbb E[Y] = \mu^2 + \sigma^2$, enquanto a propagação linear dá $\mu^2$. Para que razão $\sigma/\mu$ a diferença passa de 10 %?
2. Um Monte Carlo com $N$ amostras não observou nenhum evento. Mostre que o limite superior de 95 % para $p$ é aproximadamente $3/N$ (use $(1 - p)^N \approx e^{-Np}$).
3. Quantas avaliações de $f$ são precisas para calcular o gradiente de uma função de 1 000 parâmetros por diferenças centradas? E pelo modo reverso da diferenciação automática? Por que o treino de redes neurais (capítulo 9) usa o segundo?
4. Na seção 4, a propagação linear de $\ln\delta$ deu quase a mesma probabilidade que o Monte Carlo, e a de $\theta(0)$ errou feio. Explique a diferença.

**Verificação (com agente)**

5. Peça a um agente o cálculo da probabilidade de a pilha pegar fogo por hipercubo latino (`scipy.stats.qmc.LatinHypercube`) e compare a variabilidade da estimativa com a do Monte Carlo simples, para o mesmo $N$, repetindo cada estudo 50 vezes.
6. Peça a um agente para propagar a incerteza de $u$, $D$ e da massa derramada no problema do rio (capítulo 15) até o pico na captação, por Monte Carlo e por propagação linear. Compare as probabilidades de passar de 10 mg/L. Por que, aqui, as duas concordam?
7. Peça a um agente os índices de Sobol de primeira ordem de $\delta$ (com a biblioteca SALib ou com o estimador de Saltelli) e compare com as frações de variância da seção 5.1.

**Projeto aberto**

8. **Política de estocagem.** A temperatura ambiente varia ao longo do ano, e a pilha é montada aos poucos. Modele $T_0$ como função do mês (com variabilidade diária) e proponha uma tabela de altura máxima da pilha por mês que mantenha o risco anual de incêndio abaixo de 1 %, com o intervalo de confiança da estimativa e um teste de sensibilidade às hipóteses de distribuição.

## Leituras

- R. C. Smith, *Uncertainty Quantification: Theory, Implementation, and Applications*, SIAM, 2014.
- A. Saltelli e outros, *Global Sensitivity Analysis: The Primer*, Wiley, 2008.
- A. Griewank e A. Walther, *Evaluating Derivatives: Principles and Techniques of Algorithmic Differentiation*, 2ª ed., SIAM, 2008.
- D. A. Frank-Kamenetskii, *Diffusion and Heat Transfer in Chemical Kinetics*, Plenum, 1969 (o modelo de autoaquecimento).